In [ ]:
import torch
import numpy as np
import umap
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModel
from funcs import *

In [ ]:
# ── 1. LOAD FROM HUGGINGFACE HUB ──────────────────────────────────────────────

MODEL_NAME = "microsoft/codebert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)
model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 53381.08it/s]


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=Tru

In [38]:
raw_code = "import sys\nfrom math import sqrt, gcd, ceil, log\n# from bisect import bisect, bisect_left\nfrom collections import defaultdict, Counter, deque\n# from heapq import heapify, heappush, heappop\ninput = sys.stdin.readline\nread = lambda: list(map(int, input().strip().split()))\n\nsys.setrecursionlimit(200000)\n\n\ndef main(): \n\tn = int(input()); \n\tadj = defaultdict(list)\n\tproblem = set()\n\tfor i in range(n-1):\n\t\tx, y, t = read()\n\t\tadj[x].append(y)\n\t\tadj[y].append(x)\n\t\tif t == 2:problem.add((x, y))\n\n\n\tparent = defaultdict(int)\n\torder = []\n\tdef dfs():\n\t\tstk = [(1, 0)]\n\t\twhile stk:\n\t\t\tnode, par = stk.pop()\n\t\t\torder.append(node); parent[node] = par\n\t\t\tfor child in adj[node]:\n\t\t\t\tif child != par:\n\t\t\t\t\tstk.append((child, node))\n\t\t# return(order)\n\t\t# lis = []\n\t\t# for child in adj[node]:\n\t\t# \tif child != par:\n\t\t# \t\ttem = dfs(child, node)\n\t\t# \t\tif (node, child) in problem or (child, node) in problem:\n\t\t# \t\t\tif tem == []:lis.append(child)\n\t\t# \t\t\telse:lis.extend(tem)\n\t\t# \t\telif tem:\n\t\t# \t\t\tlis.extend(tem)\n\t\t# return(lis)\n\tdfs()\n\t# print(order)\n\t# print(parent)\n\tdic = defaultdict(int)\n\tans = []\n\tfor i in range(n-1, -1, -1):\n\t\tchild = order[i]; par = parent[order[i]]\n\t\tif dic[child]:\n\t\t\tdic[par] += dic[child]\n\t\telif (child, par) in problem or (par, child) in problem:\n\t\t\tans.append(child)\n\t\t\tdic[par] += 1\n\tprint(len(ans))\n\tprint(*ans)\n\n\n\n\t\t\t\n\n\n\n\n\n\nif __name__ == \"__main__\":\n\tmain()"
print(raw_code)

print("---------------------------------------------------------------------------------------------------------------------")

print("cleaned code : \n ")
print(preprocess_code(raw_code))

import sys
import os
from io import BytesIO
sys.stdin = BytesIO(os.read(0, os.fstat(0).st_size))
input = lambda: sys.stdin.readline().rstrip()

N,C = map(int, input().split())
A = [0]*(C+1)
for _ in range(N):
	c,d,h = map(int, input().split())
	A[c] = max(A[c], h*d)

# 与上面的循环分开，避免有多个c==1的情况
for c in range(C//2,0,-1):
	t = A[c]
	if t>0:
		for i in range(2,C//c+1):
			A[c*i] = max(A[c*i], t*i)

M = int(input())
ans = [-1]*M
B = []
for i in range(M):
	d1,h1 = map(int, input().split())
	B.append((d1*h1,i))
B.sort(reverse=True)

for c in range(1,C+1):
	e = A[c]
	while B and e>B[-1][0]:
		e1,i = B.pop()
		ans[i] = c

print(*ans)



---------------------------------------------------------------------------------------------------------------------
cleaned code : 
 
import sys
import os
from io import BytesIO
sys.stdin = BytesIO(os.read(0, os.fstat(0).st_size))
input = lambda: sys.stdin.readline().rstrip()

N,C = map(int, input().split())
A = [0]*(C+1)
for _ in range(N):
    c,d,h = map(int, 

In [ ]:
list_of_lists = [fetch_files_per_label(tag, N=256) for tag in CHOSEN_LABELS]

embeddings, labels = extract_all_labels(
    list_of_lists = list_of_lists,
    data_dir      = DATA_DIR,
    tokenizer     = tokenizer,
    model         = model,
    device        = device,
    domain        = "source_code",
    max_length    = 512,
    save_dir      = "embeddings_CodeBERT",
    batch_size    = 64
)

In [ ]:
embeddings = np.load("embeddings_CodeBERT/embeddings.npy")
labels     = np.load("embeddings_CodeBERT/labels.npy")

In [68]:
# # ── 6. PCA / UMAP + PLOT ──────────────────────────────────────────────────────
# import umap
# def plot_projection(embeddings:   np.ndarray,
#                     labels:       np.ndarray,
#                     label_names:  list  = None,
#                     method:       str   = "pca",   # "pca" or "umap"
#                     n_components: int   = 2,        # 2 or 3
#                     save_path:    str   = None):
#     """
#     Project embeddings to 2D or 3D and plot with one color per label.

#     Args:
#         embeddings:   (total_samples, 768)
#         labels:       (total_samples,)  integer label ids
#         label_names:  optional list of string names per label
#                       e.g. ["dp", "greedy", "math", ...]
#         method:       "pca" or "umap"
#         n_components: 2 for 2D plot, 3 for 3D plot
#         save_path:    if provided, saves the figure to this path
#     """
#     assert n_components in (2, 3), "n_components must be 2 or 3"
#     assert method in ("pca", "umap"), "method must be 'pca' or 'umap'"


#     # ── Normalize before projection ──
#     # PCA and UMAP are both sensitive to scale
#     print(f"Standardizing embeddings...")
#     scaler     = StandardScaler()
#     embeddings = scaler.fit_transform(embeddings)

#     # ── Dimensionality reduction ──
#     print(f"Running {method.upper()} → {n_components}D ...")

#     if method == "pca":
#         reducer  = PCA(n_components=n_components, random_state=42)
#         projected = reducer.fit_transform(embeddings)

#         explained = reducer.explained_variance_ratio_
#         print(f"Explained variance: {explained.round(3)}")
#         print(f"Total explained:    {explained.sum():.3f}")

#     else:   # umap
#         reducer  = umap.UMAP(n_components=n_components,
#                               random_state=42,
#                               n_neighbors=15,
#                               min_dist=0.1)
#         projected = reducer.fit_transform(embeddings)

#     # ── Colors — one per label ──
#     unique_labels = sorted(np.unique(labels))
#     n_labels      = len(unique_labels)
#     colors        = cm.tab10(np.linspace(0, 1, n_labels))

#     if label_names is None:
#         label_names = [f"Label {i}" for i in unique_labels]

#     # ── Plot ──
#     fig = plt.figure(figsize=(12, 8))

#     if n_components == 2:
#         ax = fig.add_subplot(111)

#         for i, label_id in enumerate(unique_labels):
#             mask = labels == label_id
#             ax.scatter(
#                 projected[mask, 0],
#                 projected[mask, 1],
#                 c=[colors[i]],
#                 label=label_names[i],
#                 alpha=0.6,
#                 s=20,
#                 edgecolors='none'
#             )

#         ax.set_xlabel("Component 1")
#         ax.set_ylabel("Component 2")

#     else:   # 3D
#         ax = fig.add_subplot(111, projection='3d')

#         for i, label_id in enumerate(unique_labels):
#             mask = labels == label_id
#             ax.scatter(
#                 projected[mask, 0],
#                 projected[mask, 1],
#                 projected[mask, 2],
#                 c=[colors[i]],
#                 label=label_names[i],
#                 alpha=0.6,
#                 s=20,
#                 edgecolors='none'
#             )

#         ax.set_xlabel("Component 1")
#         ax.set_ylabel("Component 2")
#         ax.set_zlabel("Component 3")

#     title = f"{method.upper()} {n_components}D — CodeBERT Embeddings by Label"
#     ax.set_title(title, fontsize=14, fontweight='bold')
#     ax.legend(loc='best', fontsize=9, markerscale=2)

#     plt.tight_layout()

#     if save_path:
#         plt.savefig(save_path, dpi=150, bbox_inches='tight')
#         print(f"Plot saved → {save_path}")

#     plt.show()



import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import umap

import plotly.express as px


def plot_projection(
    embeddings: np.ndarray,
    labels: np.ndarray,
    label_names: list = None,
    method: str = "pca",
    n_components: int = 2,
    save_path: str = None,
):
    """
    Interactive PCA / UMAP visualization with Plotly.
    """

    assert n_components in (2, 3)
    assert method in ("pca", "umap")

    print("Standardizing embeddings...")
    embeddings = StandardScaler().fit_transform(embeddings)

    print(f"Running {method.upper()} → {n_components}D ...")

    if method == "pca":
        reducer = PCA(
            n_components=n_components,
            random_state=42
        )

        projected = reducer.fit_transform(embeddings)

        explained = reducer.explained_variance_ratio_
        print(f"Explained variance: {explained.round(3)}")
        print(f"Total explained:    {explained.sum():.3f}")

    else:
        reducer = umap.UMAP(
            n_components=n_components,
            random_state=42,
            n_neighbors=15,
            min_dist=0.1,
        )

        projected = reducer.fit_transform(embeddings)

    unique_labels = sorted(np.unique(labels))

    if label_names is None:
        label_names = [f"Label {i}" for i in unique_labels]

    label_map = {
        lbl: label_names[i]
        for i, lbl in enumerate(unique_labels)
    }

    label_strings = [
        label_map[l]
        for l in labels
    ]

    # --------------------------
    # Build dataframe
    # --------------------------

    if n_components == 2:

        df = pd.DataFrame(
            {
                "x": projected[:, 0],
                "y": projected[:, 1],
                "label": label_strings,
            }
        )

        fig = px.scatter(
            df,
            x="x",
            y="y",
            color="label",
            title=f"{method.upper()} 2D — CodeBERT Embeddings by Label",
            opacity=0.5,      # <-- lower opacity
            hover_data=["label"],
        )

        fig.update_traces(
            marker=dict(
                size=7,
            )
        )

    else:

        df = pd.DataFrame(
            {
                "x": projected[:, 0],
                "y": projected[:, 1],
                "z": projected[:, 2],
                "label": label_strings,
            }
        )

        fig = px.scatter_3d(
            df,
            x="x",
            y="y",
            z="z",
            color="label",
            title=f"{method.upper()} 3D — CodeBERT Embeddings by Label",
            opacity=0.5,      # <-- even lower for 3D
            hover_data=["label"],
        )

        fig.update_traces(
            marker=dict(
                size=6,
            )
        )

    fig.update_layout(
        template="plotly_white",
        height=800,
        width=1200,
        legend_title="Label",
    )

    if save_path:
        if save_path.endswith(".html"):
            fig.write_html(save_path)
        else:
            fig.write_image(save_path)

        print(f"Plot saved → {save_path}")

    fig.show(renderer="browser")

In [74]:

# ── Step 3: PCA 2D ──
plot_projection(
    embeddings   = embeddings,
    labels       = labels,
    label_names  = CHOSEN_LABELS,
    method       = "pca",
    n_components = 2,
)




Standardizing embeddings...
Running PCA → 2D ...
Explained variance: [0.319 0.147]
Total explained:    0.466


In [72]:

# ── Step 3: PCA 2D ──
plot_projection(
    embeddings   = embeddings,
    labels       = labels,
    label_names  = CHOSEN_LABELS,
    method       = "pca",
    n_components = 3,
)




Standardizing embeddings...
Running PCA → 3D ...
Explained variance: [0.319 0.147 0.079]
Total explained:    0.545


In [70]:
# ── Step 4: UMAP 3D ──
plot_projection(
    embeddings   = embeddings,
    labels       = labels,
    label_names  = CHOSEN_LABELS,
    method       = "umap",
    n_components = 3,
)


Standardizing embeddings...
Running UMAP → 3D ...


c:\Users\kajou\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [71]:
# ── Step 4: UMAP 3D ──
plot_projection(
    embeddings   = embeddings,
    labels       = labels,
    label_names  = CHOSEN_LABELS,
    method       = "umap",
    n_components = 2,
)


Standardizing embeddings...
Running UMAP → 2D ...


c:\Users\kajou\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [73]:
# Add this debug block in your pipeline
from collections import Counter

# hash each embedding row to find true duplicates
hashes = [hash(e.tobytes()) for e in embeddings]
counts = Counter(hashes)
true_duplicates = {h: c for h, c in counts.items() if c > 1}
print(f"True duplicate embeddings: {len(true_duplicates)} groups")

True duplicate embeddings: 185 groups
